In [ ]:
# =============================================================
# Sleep Quality Scorer - runs as ONE Colab cell (or as a script)
# =============================================================
import json
import numpy as np
import pandas as pd

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


class SleepScoreAgent:

    WEIGHTS = {
        "Heart_Rate_Variability": 0.20,
        "Body_Temperature": 0.10,
        "Movement_During_Sleep": 0.15,
        "Sleep_Duration_Hours": 0.20,
        "Caffeine_Intake_mg": 0.10,
        "Stress_Level": 0.15,
        "Bedtime_Consistency": 0.10,
    }

    IDEAL_TEMP_C = 36.5
    DURATION_SWEET_SPOT = (7, 9)

    def __init__(self):
        self.bounds_ = {}
        self.is_fit = False

    # ---------------------------------------------------------
    # Clean CSV column names
    # ---------------------------------------------------------
    @staticmethod
    def clean_columns(df):
        df = df.copy()
        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(" ", "_", regex=False)
            .str.replace("-", "_", regex=False)
        )
        return df

    # ---------------------------------------------------------
    # Rename common column variations
    # ---------------------------------------------------------
    @staticmethod
    def standardize_columns(df):
        column_mapping = {
            # HRV
            "hrv": "Heart_Rate_Variability",
            "heart_rate_variability": "Heart_Rate_Variability",
            "heart_rate_variability_ms": "Heart_Rate_Variability",
            "heart_rate_var": "Heart_Rate_Variability",
            # Temperature
            "body_temperature": "Body_Temperature",
            "body_temperature_c": "Body_Temperature",
            "temperature": "Body_Temperature",
            "temp": "Body_Temperature",
            # Movement
            "movement_during_sleep": "Movement_During_Sleep",
            "movement": "Movement_During_Sleep",
            "sleep_movement": "Movement_During_Sleep",
            # Sleep duration
            "sleep_duration_hours": "Sleep_Duration_Hours",
            "sleep_duration": "Sleep_Duration_Hours",
            "duration": "Sleep_Duration_Hours",
            "sleep_hours": "Sleep_Duration_Hours",
            # Caffeine
            "caffeine_intake_mg": "Caffeine_Intake_mg",
            "caffeine": "Caffeine_Intake_mg",
            "caffeine_mg": "Caffeine_Intake_mg",
            # Stress
            "stress_level": "Stress_Level",
            "stress": "Stress_Level",
            # Bedtime consistency
            "bedtime_consistency": "Bedtime_Consistency",
            "bedtime_consistency_ratio": "Bedtime_Consistency",
            "sleep_consistency": "Bedtime_Consistency",
        }
        return df.rename(columns=column_mapping)

    # ---------------------------------------------------------
    # Fit model
    # ---------------------------------------------------------
    def fit(self, df):
        for col in self.WEIGHTS:
            if col not in df.columns:
                raise ValueError(f"Required column '{col}' not found in dataset.")
            df[col] = pd.to_numeric(df[col], errors="coerce")
            self.bounds_[col] = (df[col].min(), df[col].max())
        self.is_fit = True
        return self

    # ---------------------------------------------------------
    # Min-Max normalization
    # ---------------------------------------------------------
    def _minmax(self, series, col, higher_is_better=True):
        lo, hi = self.bounds_[col]
        rng = max(hi - lo, 1e-9)
        scaled = (series - lo) / rng * 100
        if not higher_is_better:
            scaled = 100 - scaled
        return scaled.clip(0, 100)

    # ---------------------------------------------------------
    # Sleep duration score
    # ---------------------------------------------------------
    def _duration_score(self, hours):
        lo, hi = self.DURATION_SWEET_SPOT
        score = pd.Series(100.0, index=hours.index)

        below = hours < lo
        above = hours > hi

        d_lo, d_hi = self.bounds_["Sleep_Duration_Hours"]
        span_low = max(lo - d_lo, 1e-9)
        span_high = max(d_hi - hi, 1e-9)

        score.loc[below] = 100 * (1 - (lo - hours.loc[below]) / span_low)
        score.loc[above] = 100 * (1 - (hours.loc[above] - hi) / span_high)

        return score.clip(0, 100)

    # ---------------------------------------------------------
    # Body temperature score
    # ---------------------------------------------------------
    def _temp_score(self, temp):
        t_lo, t_hi = self.bounds_["Body_Temperature"]
        max_dev = max(
            abs(t_hi - self.IDEAL_TEMP_C),
            abs(self.IDEAL_TEMP_C - t_lo),
            1e-9,
        )
        score = 100 * (1 - (temp - self.IDEAL_TEMP_C).abs() / max_dev)
        return score.clip(0, 100)

    # ---------------------------------------------------------
    # Score dataset
    # ---------------------------------------------------------
    def score(self, df):
        if not self.is_fit:
            raise RuntimeError("Call .fit(reference_df) before .score().")

        out = df.copy()
        W = self.WEIGHTS

        out["HRV_Score"] = self._minmax(
            df["Heart_Rate_Variability"], "Heart_Rate_Variability", True)
        out["Temp_Score"] = self._temp_score(df["Body_Temperature"])
        out["Movement_Score"] = self._minmax(
            df["Movement_During_Sleep"], "Movement_During_Sleep", False)
        out["Duration_Score"] = self._duration_score(df["Sleep_Duration_Hours"])
        out["Caffeine_Score"] = self._minmax(
            df["Caffeine_Intake_mg"], "Caffeine_Intake_mg", False)
        out["Stress_Score"] = self._minmax(
            df["Stress_Level"], "Stress_Level", False)
        out["Bedtime_Score"] = self._minmax(
            df["Bedtime_Consistency"], "Bedtime_Consistency", True)

        out["Sleep_Quality_Score"] = (
            W["Heart_Rate_Variability"] * out["HRV_Score"]
            + W["Body_Temperature"] * out["Temp_Score"]
            + W["Movement_During_Sleep"] * out["Movement_Score"]
            + W["Sleep_Duration_Hours"] * out["Duration_Score"]
            + W["Caffeine_Intake_mg"] * out["Caffeine_Score"]
            + W["Stress_Level"] * out["Stress_Score"]
            + W["Bedtime_Consistency"] * out["Bedtime_Score"]
        ).round(2)

        out["Sleep_Category"] = pd.cut(
            out["Sleep_Quality_Score"],
            bins=[-0.1, 39, 59, 74, 89, 100],
            labels=["Very Poor", "Poor", "Fair", "Good", "Excellent"],
        )
        return out


# =============================================================
# Helpers
# =============================================================
def load_data(path):
    """Load a CSV, or a JSON object / list of objects."""
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    if text.lstrip()[:1] in ("{", "["):
        data = json.loads(text)
        return pd.DataFrame([data] if isinstance(data, dict) else data)
    return pd.read_csv(path)


def run(path, output="wearable_tech_sleep_quality_scored.csv"):
    df = load_data(path)
    df = SleepScoreAgent.standardize_columns(SleepScoreAgent.clean_columns(df))

    print("Columns detected:", list(df.columns))

    required = list(SleepScoreAgent.WEIGHTS)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=required).reset_index(drop=True)
    if df.empty:
        raise ValueError("No valid rows remain after cleaning.")

    agent = SleepScoreAgent().fit(df)
    scored = agent.score(df)
    scored.to_csv(output, index=False)

    print("\n========================================")
    print("       SLEEP QUALITY SCORE RESULTS")
    print("========================================")
    print("\nRows scored:", len(scored))
    print("\nScore distribution:")
    print(scored["Sleep_Quality_Score"].describe().round(2))
    print("\nCategory counts:")
    print(scored["Sleep_Category"].value_counts())
    print("\nSample results:")
    print(
        scored[["Heart_Rate_Variability", "Sleep_Duration_Hours", "Stress_Level",
                "Sleep_Quality_Score", "Sleep_Category"]]
        .head(10).to_string(index=False)
    )
    print(f"\nSaved scored file to: {output}")
    return scored, output


# =============================================================
# Run
# =============================================================
if IN_COLAB:
    uploaded = files.upload()               # choose sleep_dataset.csv
    input_path = next(iter(uploaded))
else:
    input_path = "sleep_dataset.csv"        # local fallback

scored, out_path = run(input_path)

if IN_COLAB:
    files.download(out_path)

Saving sleep_dataset.csv to sleep_dataset (1).csv
Columns detected: ['Heart_Rate_Variability', 'Body_Temperature', 'Movement_During_Sleep', 'Sleep_Duration_Hours', 'Caffeine_Intake_mg', 'Stress_Level', 'Bedtime_Consistency', 'light_exposure_hours']

       SLEEP QUALITY SCORE RESULTS

Rows scored: 1000

Score distribution:
count    1000.00
mean       61.20
std         7.05
min        31.63
25%        56.58
50%        61.66
75%        66.18
max        79.97
Name: Sleep_Quality_Score, dtype: float64

Category counts:
Sleep_Category
Fair         610
Poor         359
Good          26
Very Poor      5
Excellent      0
Name: count, dtype: int64

Sample results:
 Heart_Rate_Variability  Sleep_Duration_Hours  Stress_Level  Sleep_Quality_Score Sleep_Category
              79.934283              4.638289      2.771837                58.69           Poor
              67.234714              6.209422      3.738138                58.80           Poor
              82.953771              6.879592   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>